In [ ]:
import numpy as np

def simulasi_bengkel(num_mobil=50, mean_iat=12, mean_repair=15, std_repair=3, mean_reg=3, mean_cashier=3, seed=42):
    np.random.seed(seed)
    
    # 1. Inisialisasi Variabel
    clock = 0.0
    total_mechanics = 2
    mechanics_busy = 0
    waiting_queue = [] # Antrian di ruang tunggu
    event_calendar = [] # Format: [waktu_kejadian, tipe_kejadian, car_id]
    
    # Bangkitkan waktu acak menggunakan distribusi probabilitas
    waktu_antar_kedatangan = np.random.exponential(mean_iat, num_mobil - 1)
    waktu_reg = np.random.normal(mean_reg, 1, num_mobil)
    waktu_perbaikan = np.random.normal(mean_repair, std_repair, num_mobil)
    waktu_kasir = np.random.normal(mean_cashier, 1, num_mobil)
    
    # Pastikan durasi bernilai positif
    waktu_reg = [max(1, x) for x in waktu_reg]
    waktu_perbaikan = [max(3, x) for x in waktu_perbaikan]
    waktu_kasir = [max(1, x) for x in waktu_kasir]
    
    arrival_times = {}
    car_data = {}
    
    for i in range(num_mobil):
        car_id = i + 1
        car_data[car_id] = {
            'reg_time': waktu_reg[i],
            'repair_time': waktu_perbaikan[i],
            'cashier_time': waktu_kasir[i]
        }

    # Jadwalkan kedatangan mobil pertama pada clock 0.0
    event_calendar.append([0.0, "ARRIVAL", 1])
    arrival_times[1] = 0.0
    
    # Jadwalkan kedatangan mobil-mobil berikutnya
    next_arrival = 0.0
    for i in range(len(waktu_antar_kedatangan)):
        next_arrival += waktu_antar_kedatangan[i]
        car_id = i + 2
        arrival_times[car_id] = next_arrival
        event_calendar.append([next_arrival, "ARRIVAL", car_id])
    
    # Urutkan Event Calendar berdasarkan waktu
    event_calendar.sort(key=lambda x: x[0])
    
    # Variabel Statistik
    total_wait_time = 0.0
    max_wait_time = 0.0
    cars_served = 0
    
    print("=== SIMULASI SISTEM BENGKEL (2 MONTIR) ===\n")
    print(f"{'Clock':<10}{'Event':<20}{'Car ID':<10}{'Queue':<10}{'Busy Mech':<12}{'Keterangan':<25}")
    print("-" * 90)
    
    # 2. Main Simulation Loop
    while event_calendar:
        current_event = event_calendar.pop(0)
        clock, event_type, car_id = current_event
        clock = round(clock, 2)
        
        if event_type == "ARRIVAL":
            print(f"{clock:<10.2f}{'ARRIVAL':<20}{car_id:<10}{len(waiting_queue):<10}{mechanics_busy:<12}{'Datang & Pendaftaran':<25}")
            
            # Pengecekan Ketersediaan Montir
            if mechanics_busy < total_mechanics:
                mechanics_busy += 1
                rep_time = car_data[car_id]['repair_time']
                dep_repair_time = clock + rep_time
                event_calendar.append([dep_repair_time, "REPAIR_COMPLETE", car_id])
                event_calendar.sort(key=lambda x: x[0])
                print(f"{clock:<10.2f}{'START_REPAIR':<20}{car_id:<10}{len(waiting_queue):<10}{mechanics_busy:<12}{'Montir Tersedia (Mulai)':<25}")
            else:
                # Masuk Ruang Tunggu jika kedua montir sibuk
                waiting_queue.append(car_id)
                print(f"{clock:<10.2f}{'WAITING_ROOM':<20}{car_id:<10}{len(waiting_queue):<10}{mechanics_busy:<12}{'Montir Sibuk, Masuk Antri':<25}")
                
        elif event_type == "REPAIR_COMPLETE":
            mechanics_busy -= 1
            print(f"{clock:<10.2f}{'REPAIR_COMPLETE':<20}{car_id:<10}{len(waiting_queue):<10}{mechanics_busy:<12}{'Perbaikan Selesai':<25}")
            
            # Jika ada antrian di ruang tunggu, ambil mobil berikutnya
            if waiting_queue:
                next_car_id = waiting_queue.pop(0)
                mechanics_busy += 1
                
                # Hitung waktu tunggu
                wait_time = clock - arrival_times[next_car_id]
                total_wait_time += wait_time
                max_wait_time = max(max_wait_time, wait_time)
                
                rep_time = car_data[next_car_id]['repair_time']
                dep_repair_time = clock + rep_time
                event_calendar.append([dep_repair_time, "REPAIR_COMPLETE", next_car_id])
                event_calendar.sort(key=lambda x: x[0])
                print(f"{clock:<10.2f}{'PULL_FROM_QUEUE':<20}{next_car_id:<10}{len(waiting_queue):<10}{mechanics_busy:<12}{f'Ambil Antrian (Wait: {wait_time:.1f}m)':<25}")
            
            # Lanjut ke proses pembayaran kasir
            cashier_time = car_data[car_id]['cashier_time']
            cashier_complete_time = clock + cashier_time
            event_calendar.append([cashier_complete_time, "CASHIER_COMPLETE", car_id])
            event_calendar.sort(key=lambda x: x[0])
            
        elif event_type == "CASHIER_COMPLETE":
            cars_served += 1
            print(f"{clock:<10.2f}{'CASHIER_COMPLETE':<20}{car_id:<10}{len(waiting_queue):<10}{mechanics_busy:<12}{'Bayar & Keluar Bengkel':<25}")

    # 3. Kalkulasi Statistik Akhir
    avg_wait_time = total_wait_time / cars_served if cars_served > 0 else 0.0
    print("\n" + "="*50)
    print("=== HASIL STATISTIK SIMULASI BENGKEL ===")
    print(f"Total mobil dilayani       : {cars_served}")
    print(f"Rata-rata waktu tunggu     : {avg_wait_time:.2f} menit")
    print(f"Waktu tunggu maksimum      : {max_wait_time:.2f} menit")
    print(f"Total waktu simulasi       : {clock:.2f} menit")
    print("="*50)

# Jalankan simulasi
simulasi_bengkel()